# Frozen Qwen3.8 LoRA replication

The frozen mixed-source adapter is compared with the same base model on a separately audited 25-task exploratory set. Unknown semantic and pretraining exposure remain. The earlier, already-examined 50-task result was 2/50 to 5/50 on either recorded candidate; it is a different dataset, not the new-set baseline. This is cross-run distillation, not demonstrated recursive improvement.


In [ ]:
from pathlib import Path
import json, os
from src.arc.qwen38_rsi_comparison import digest

repo = Path.cwd().resolve()
while not (repo / 'research/rsi/qwen38-replication-v1.json').is_file():
    if repo == repo.parent:
        raise FileNotFoundError('Run inside the issue 18 worktree')
    repo = repo.parent
registration = json.loads((repo / 'research/rsi/qwen38-replication-v1.json').read_text())
if registration.get('sha256') != digest({k: v for k, v in registration.items() if k != 'sha256'}):
    raise ValueError('registration changed')
run = Path(os.environ.get('QWEN38_REPLICATION_RUN_DIR', repo / 'artifacts/qwen38-replication-v1'))
print('Registered tasks:', len(registration['task_ids']), 'designation:', registration['designation'])
print('Registration:', registration['sha256'])


## Matched results

Generate the report only after the registered run terminates and its receipts have been replayed by the offline CLI. A missing or invalid report is not a zero score. The display uses local receipt summaries and does not expose candidate text or answer grids. Two recorded candidates are not assumed independent.


In [ ]:
reports = []
for path in run.glob('report-*.json'):
    try:
        value = json.loads(path.read_text())
        if (value.get('registration_sha256') == registration['sha256'] and
            value.get('sha256') == digest({k: v for k, v in value.items() if k != 'sha256'})):
            reports.append((path.stat().st_mtime_ns, value))
    except (OSError, ValueError, TypeError):
        continue
if not reports:
    print('Report not available; evaluation is not scored here.')
else:
    report = max(reports, key=lambda item: item[0])[1]
    print('Report status:', report.get('status'))
    print('Report identity:', report.get('sha256'))
    paired = report.get('paired', {})
    arm_fields = ('requested_tasks', 'complete_tasks', 'incomplete_tasks', 'missing_tasks', 'failed_tasks',
                  'partial_tasks', 'first_correct', 'either_correct', 'valid_candidates', 'invalid_candidates',
                  'duplicate_raw_pairs', 'runtime_seconds')
    for arm in ('baseline', 'gen1'):
        summary = paired.get(arm, {})
        print(arm, {key: summary[key] for key in arm_fields if key in summary})
    counts = paired.get('paired', {})
    print('Paired:', {key: counts[key] for key in ('complete', 'gained', 'lost', 'both_correct', 'both_incorrect') if key in counts})
    uncertainty = report.get('uncertainty')
    if isinstance(uncertainty, dict):
        print('Exact sign test:', {key: uncertainty[key] for key in ('discordant', 'two_sided_p') if key in uncertainty})
    print('Inference errors:', report.get('errors', {}))
    print('Run wall seconds:', report.get('wall_seconds'))
